# Oil Full Classifier Final Clean

This clean notebook follows the final MAG7 full-classifier structure for the oil conflict-transfer study. It keeps oil-specific data loading, return windows, and conflict context, while using the same training, validation, test, frozen-rule, ablation, and repeated-seed structure.


## 1. Aim And Setup

This notebook evaluates a full-event oil price-direction classifier. Each retained conflict-news event is classified as realised bullish, bearish, or neutral for oil. The terminology uses **training**, **validation**, and **test** throughout.


In [1]:
from __future__ import annotations

import json
from itertools import combinations
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import binomtest

import matplotlib.pyplot as plt
import seaborn as sns

import ltn

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

# Use the folder the notebook is currently running from.
WORK_DIR = Path.cwd()

# Oil case-study event-return file saved beside this notebook.
# This replaces MAG7's refined_event_returns.parquet, mag7_daily_prices.parquet,
# and sp500_daily_prices.parquet setup.
SOURCE_PATH = WORK_DIR / "aligned_oil_event_returns.parquet"

required_files = {
    "aligned_oil_event_returns.parquet": SOURCE_PATH,
}

missing = [name for name, path in required_files.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required file(s) in the current folder:\n"
        + "\n".join(f"- {name}" for name in missing)
        + f"\n\nCurrent folder is:\n{WORK_DIR}"
    )

# Write all outputs to the current folder.
OUTPUT_DIR = WORK_DIR / "16_oil_full_classifier_final_clean_ret_24h"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Oil already has event-window return columns. RETURN_COL is the retained primary return horizon.
RETURN_COL = "ret_24h"
RETURN_WINDOW_CANDIDATES = [
    ("1h", "ret_1h"),
    ("2h", "ret_2h"),
    ("3h", "ret_3h"),
    ("4h", "ret_4h"),
    ("6h", "ret_6h"),
    ("8h", "ret_8h"),
    ("24h", "ret_24h"),
]
RETURN_THRESHOLD = 0.0

# Oil uses a conflict-transfer split, created in the next loading cell:
# - Russia/Ukraine events -> training + validation
# - Iran events -> held-out transfer test
TRAINING_VALIDATION_SOURCE = "russia_ukraine_war"
TEST_SOURCE = "iran_war"
TRAINING_FRACTION = 0.60

EPOCHS = 500
LR = 0.01
LOGIC_WEIGHT = 0.30
SEED = 7
ROBUST_SEEDS = [1, 2, 3, 4, 5, 7, 11, 13, 17, 19]

LABELS = ["bearish", "bullish", "neutral"]

ANTECEDENT_THRESHOLD = 0.50
MIN_TRAINING_EVENTS = 6
MIN_VALIDATION_EVENTS = 5
MIN_TRAINING_ACCURACY = 60.0
MIN_VALIDATION_ACCURACY = 60.0

SAVE_OUTPUTS = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("working folder:", WORK_DIR)
print("source:", SOURCE_PATH)
print("output:", OUTPUT_DIR)
print("return column:", RETURN_COL)
print("split:", TRAINING_VALIDATION_SOURCE, "training/validation ->", TEST_SOURCE, "test")
print("ltn:", getattr(ltn, "__version__", "unknown"))
print("torch:", torch.__version__, "device:", device)

working folder: /home/jovyan/Stock-Sentiment-Prediction
source: /home/jovyan/Stock-Sentiment-Prediction/aligned_oil_event_returns.parquet
output: /home/jovyan/Stock-Sentiment-Prediction/16_oil_full_classifier_final_clean_ret_24h
return column: ret_24h
split: russia_ukraine_war training/validation -> iran_war test
ltn: unknown
torch: 2.12.1+cu126 device: cuda


In [2]:
def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

reset_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print("Could not force all deterministic algorithms:", e)

## 2. Load Event And Price Data

This section loads the aligned oil event-return table. Oil returns are already present as `ret_*` columns, so the notebook defines the label function and constructs the Russia/Ukraine training-validation split plus the Iran held-out test split.


In [3]:
# OIL DIFFERENCE:
# MAG7 loads event rows plus separate stock/SP500 price files.
# Oil uses one aligned event-return file that already contains:
# - Qwen/geopolitical event predicates
# - oil return columns such as ret_4h and ret_24h
# - pre-event oil context columns such as ret_pre_1h, ret_pre_4h, ret_pre_24h

events_all = pd.read_parquet(SOURCE_PATH).copy()

events_all["publication_timestamp"] = pd.to_datetime(
    events_all["publication_timestamp"],
    utc=True,
)

# OIL DIFFERENCE:
# MAG7 computes simple_return from stock prices and PRIMARY_HORIZON.
# Oil already has return horizons as columns, e.g. ret_4h and ret_24h.
# We still use the same label logic: positive return = bullish, negative return = bearish,
# and values inside the threshold band = neutral.
def realised_label_from_return(value: float, threshold: float = RETURN_THRESHOLD) -> str:
    if pd.isna(value):
        return "neutral"
    if value > threshold:
        return "bullish"
    if value < -threshold:
        return "bearish"
    return "neutral"


required_columns = [
    "conflict",
    "publication_timestamp",
    "headline",
    RETURN_COL,
    "llm_event_direction",
    "llm_direct_label",
    "mechanism_signature",
    "high_confidence",
]

missing_columns = [col for col in required_columns if col not in events_all.columns]
if missing_columns:
    raise ValueError(
        "Missing required oil column(s):\n"
        + "\n".join(f"- {col}" for col in missing_columns)
    )

events_all[RETURN_COL] = pd.to_numeric(events_all[RETURN_COL], errors="coerce")

missing_returns = int(events_all[RETURN_COL].isna().sum())
if missing_returns:
    print(f"Dropping {missing_returns} oil event rows with missing {RETURN_COL}")
    events_all = events_all.loc[events_all[RETURN_COL].notna()].copy()

events_all["target_label"] = events_all[RETURN_COL].map(
    lambda x: realised_label_from_return(float(x), RETURN_THRESHOLD)
)

# OIL DIFFERENCE:
# MAG7 uses calendar date splits.
# Oil uses a transfer-style split:
# - Russia/Ukraine events are split chronologically into training and validation.
# - Iran events are held out as the test/transfer set.
russia = (
    events_all.loc[events_all["conflict"].eq(TRAINING_VALIDATION_SOURCE)]
    .sort_values("publication_timestamp")
    .copy()
)

iran = (
    events_all.loc[events_all["conflict"].eq(TEST_SOURCE)]
    .sort_values("publication_timestamp")
    .copy()
)

if russia.empty:
    raise ValueError(f"No training/validation rows found for conflict={TRAINING_VALIDATION_SOURCE!r}")

if iran.empty:
    raise ValueError(f"No test rows found for conflict={TEST_SOURCE!r}")

cut = int(len(russia) * TRAINING_FRACTION)

russia.loc[russia.index[:cut], "period"] = "training"
russia.loc[russia.index[cut:], "period"] = "validation"
iran["period"] = "test"

events = (
    pd.concat([russia, iran], ignore_index=True)
    .sort_values(["period", "publication_timestamp"])
    .reset_index(drop=True)
)

display(events.groupby("period")["target_label"].value_counts().unstack(fill_value=0))

display(events[[
    "conflict",
    "publication_timestamp",
    "headline",
    "llm_event_direction",
    "llm_direct_label",
    "target_label",
    RETURN_COL,
    "mechanism_signature",
    "high_confidence",
]].head())

target_label,bearish,bullish,neutral
period,,,
test,89,38,0
training,289,326,0
validation,220,189,1


,conflict,publication_timestamp,headline,llm_event_direction,llm_direct_label,target_label,ret_24h,mechanism_signature,high_confidence
0,iran_war,2026-05-04 13:00:00+00:00,First Thing: Trump says US navy will 'guide' t...,escalation,bullish,bullish,0.009563,escalation+supply_risk+shipping_or_hormuz,normal_conf
1,iran_war,2026-05-04 16:10:14+00:00,Shipping firms question safety in strait of Ho...,escalation,bullish,bearish,-0.035390,escalation+supply_risk+shipping_or_hormuz,normal_conf
2,iran_war,2026-05-04 19:26:25+00:00,Donald Trump sends warships to break Iran's st...,escalation,bullish,bearish,-0.034728,escalation+supply_risk+shipping_or_hormuz,high_conf
3,iran_war,2026-05-04 20:51:32+00:00,Trump threatens to blow Iran 'off the face of ...,escalation,bullish,bearish,-0.029595,escalation+supply_risk+military_action,high_conf
4,iran_war,2026-05-04 20:53:38+00:00,Morning Mail: tensions rise as Trump tries to ...,escalation,bullish,bearish,-0.029595,escalation+supply_risk+military_action,normal_conf


## 3. Define Return Horizon Before Modelling

`RETURN_COL` is declared in setup, and `RETURN_WINDOW_CANDIDATES` are available before model training. The final labelled modelling frame is intentionally delayed until after the validation-only return-window audit.


## 4. Build Horizon-Independent Context Features

This section builds oil pre-event market context, conflict-news flow, timing context, Qwen/geopolitical predicates, and interaction features.


In [4]:
# OIL DIFFERENCE:
# MAG7 builds ticker/SP500 market context from separate price files.
# Oil already has pre-event return columns in the aligned oil event-return parquet.
# Here we preserve the same structure: build a ctx dataframe containing
# event rows + pre-event market context + news-flow/timing context.

ctx = events.copy()

# OIL DIFFERENCE:
# MAG7 pre-event context used daily stock/SP500 returns.
# Oil pre-event context uses short-window oil returns already present in the data.
required_context_cols = ["ret_pre_1h", "ret_pre_4h", "ret_pre_24h"]
missing_context_cols = [col for col in required_context_cols if col not in ctx.columns]

if missing_context_cols:
    raise ValueError(
        "Missing required oil pre-event return column(s):\n"
        + "\n".join(f"- {col}" for col in missing_context_cols)
    )

for h in ["1h", "4h", "24h"]:
    ctx[f"oil_ret_{h}_pre"] = pd.to_numeric(ctx[f"ret_pre_{h}"], errors="coerce")
    ctx[f"oil_abs_ret_{h}_pre"] = ctx[f"oil_ret_{h}_pre"].abs()


# OIL DIFFERENCE:
# MAG7 groups news-flow by ticker.
# Oil groups news-flow by conflict because the case study is a conflict-transfer design:
# Russia/Ukraine training-validation -> Iran test.
ctx = ctx.sort_values(["conflict", "publication_timestamp"]).reset_index(drop=True)

ctx["event_density_conflict_6h"] = 0.0
ctx["event_density_conflict_24h"] = 0.0
ctx["similar_density_conflict_24h"] = 0.0

for conflict, group in ctx.groupby("conflict"):
    for idx, ts, mechanism in zip(
        group.index,
        group["publication_timestamp"],
        group["mechanism_signature"],
    ):
        prev = group.loc[group["publication_timestamp"].lt(ts)]

        last6 = prev.loc[
            prev["publication_timestamp"].ge(ts - pd.Timedelta(hours=6))
        ]

        last24 = prev.loc[
            prev["publication_timestamp"].ge(ts - pd.Timedelta(hours=24))
        ]

        # OIL DIFFERENCE:
        # MAG7 similarity used event_subtype or valuation_channel.
        # Oil similarity uses the extracted geopolitical mechanism signature.
        similar24 = last24.loc[
            last24["mechanism_signature"].eq(mechanism)
        ]

        ctx.loc[idx, "event_density_conflict_6h"] = len(last6)
        ctx.loc[idx, "event_density_conflict_24h"] = len(last24)
        ctx.loc[idx, "similar_density_conflict_24h"] = len(similar24)


# OIL DIFFERENCE:
# Keep cyclic hour-of-publication features for the oil event timestamp.
hour = ctx["publication_timestamp"].dt.hour + ctx["publication_timestamp"].dt.minute / 60

ctx["hour_sin_01"] = (np.sin(2 * np.pi * hour / 24) + 1) / 2
ctx["hour_cos_01"] = (np.cos(2 * np.pi * hour / 24) + 1) / 2


display(ctx[[
    "conflict",
    "publication_timestamp",
    "target_label",
    "llm_direct_label",
    RETURN_COL,
    "oil_ret_1h_pre",
    "oil_ret_4h_pre",
    "oil_ret_24h_pre",
    "event_density_conflict_6h",
    "event_density_conflict_24h",
    "similar_density_conflict_24h",
    "hour_sin_01",
    "hour_cos_01",
]].head(10))

,conflict,publication_timestamp,target_label,llm_direct_label,ret_24h,oil_ret_1h_pre,oil_ret_4h_pre,oil_ret_24h_pre,event_density_conflict_6h,event_density_conflict_24h,similar_density_conflict_24h,hour_sin_01,hour_cos_01
0,iran_war,2026-05-04 13:00:00+00:00,bullish,bullish,0.009563,-0.001261,0.014738,NaN,0.0,0.0,0.0,0.370590,0.017037
1,iran_war,2026-05-04 16:10:14+00:00,bearish,bullish,-0.035390,0.014719,0.032479,NaN,1.0,1.0,1.0,0.056495,0.269126
2,iran_war,2026-05-04 19:26:25+00:00,bearish,bullish,-0.034728,-0.002013,0.001317,NaN,1.0,2.0,2.0,0.034791,0.683251
3,iran_war,2026-05-04 20:51:32+00:00,bearish,bullish,-0.029595,-0.000965,-0.008964,NaN,2.0,3.0,0.0,0.132839,0.839400
4,iran_war,2026-05-04 20:53:38+00:00,bearish,bullish,-0.029595,-0.000965,-0.008964,NaN,3.0,4.0,1.0,0.135815,0.842591
5,iran_war,2026-05-05 00:30:37+00:00,bearish,neutral,-0.045985,-0.000351,0.002463,0.051878,3.0,5.0,0.0,0.565263,0.995722
6,iran_war,2026-05-05 01:00:37+00:00,bearish,bullish,-0.047795,0.002025,0.000527,0.057708,4.0,6.0,3.0,0.629410,0.982963
7,iran_war,2026-05-05 05:34:18+00:00,bearish,neutral,-0.043949,0.005580,-0.002109,0.056383,2.0,7.0,0.0,0.996786,0.556602
8,iran_war,2026-05-05 10:25:01+00:00,bearish,bullish,-0.109565,0.001243,-0.011133,-0.007129,1.0,8.0,2.0,0.701373,0.042344
9,iran_war,2026-05-05 16:49:42+00:00,bearish,neutral,-0.072735,0.002270,-0.015341,-0.037908,0.0,7.0,3.0,0.023802,0.347568


## 5. Create Chronological Training / Validation / Test Split

This section checks the transfer split and builds `feature_df`, which contains oil/Qwen predicates, context features, interaction features, and period labels. Final target labels are rebuilt from the selected return window after the validation-only audit.


In [5]:
# OIL DIFFERENCE:
# MAG7 assigns periods from calendar dates here.
# Oil periods were already assigned in the load cell:
# - Russia/Ukraine early rows -> training
# - Russia/Ukraine later rows -> validation
# - Iran rows -> test
#
# So this cell checks the split and then fits all feature scales/dummy categories
# using training rows only, matching the leakage-safe MAG7 final notebook.

expected_periods = {"training", "validation", "test"}
observed_periods = set(ctx["period"].dropna().astype(str))

missing_periods = expected_periods - observed_periods
if missing_periods:
    raise ValueError(f"Missing expected oil period(s): {sorted(missing_periods)}")

fit_mask = ctx["period"].eq("training")


def fit_abs_scale(col, q=0.80):
    s = pd.to_numeric(ctx.loc[fit_mask, col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    scale = s.abs().quantile(q)
    return float(scale) if np.isfinite(scale) and scale != 0 else 1.0


def fit_quantile(col, q, fallback=1.0):
    s = pd.to_numeric(ctx.loc[fit_mask, col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    value = s.quantile(q)
    return float(value) if np.isfinite(value) and value != 0 else fallback


def soft_pos_with_scale(s, scale):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return (s.clip(lower=0) / scale).clip(0, 1).fillna(0)


def soft_neg_with_scale(s, scale):
    return soft_pos_with_scale(-pd.to_numeric(s, errors="coerce"), scale)


def soft_abs_with_scale(s, scale):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return (s.abs() / scale).clip(0, 1).fillna(0)


def soft_low_with_cutoff(s, cutoff):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return (1 - (s / cutoff)).clip(0, 1).fillna(0)


def soft_high_with_cutoff(s, cutoff):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return (s / cutoff - 1).clip(0, 1).fillna(0)


# OIL DIFFERENCE:
# MAG7 used ticker/SP500 5-day pre-event returns.
# Oil uses short pre-event oil windows already present in the event-return table.
for h in ["1h", "4h", "24h"]:
    ret_col = f"oil_ret_{h}_pre"
    scale = fit_abs_scale(ret_col)

    ctx[f"prior_up_{h}"] = soft_pos_with_scale(ctx[ret_col], scale)
    ctx[f"prior_down_{h}"] = soft_neg_with_scale(ctx[ret_col], scale)
    ctx[f"prior_abs_{h}"] = soft_abs_with_scale(ctx[ret_col], scale)


# OIL DIFFERENCE:
# Quiet/volatile context is based on absolute pre-event oil movement.
# Scales are fitted only on training rows.
ctx["quiet_pre_4h"] = soft_low_with_cutoff(
    ctx["oil_abs_ret_4h_pre"],
    fit_quantile("oil_abs_ret_4h_pre", 0.40),
)
ctx["volatile_pre_4h"] = soft_high_with_cutoff(
    ctx["oil_abs_ret_4h_pre"],
    fit_quantile("oil_abs_ret_4h_pre", 0.70),
)
ctx["quiet_pre_24h"] = soft_low_with_cutoff(
    ctx["oil_abs_ret_24h_pre"],
    fit_quantile("oil_abs_ret_24h_pre", 0.40),
)
ctx["volatile_pre_24h"] = soft_high_with_cutoff(
    ctx["oil_abs_ret_24h_pre"],
    fit_quantile("oil_abs_ret_24h_pre", 0.70),
)


# OIL DIFFERENCE:
# MAG7 normalised ticker event density over 7/30 days.
# Oil normalises conflict news density over 6/24 hours.
for col in [
    "event_density_conflict_6h",
    "event_density_conflict_24h",
    "similar_density_conflict_24h",
]:
    denom = max(ctx.loc[fit_mask, col].quantile(0.95), 1)
    ctx[col + "_01"] = (ctx[col] / denom).clip(0, 1)

ctx["novel_context_24h"] = (1 - ctx["similar_density_conflict_24h_01"]).clip(0, 1)


# OIL DIFFERENCE:
# These are the oil/Qwen/geopolitical categorical predicates.
# Categories are fitted from training only, matching the MAG7 leakage-safe pattern.
categorical_cols = [
    "llm_event_direction",
    "llm_oil_price_direction",
    "llm_reaction_horizon",
    "llm_direct_label",
    "news_weak_label",
    "market_momentum_label",
    "high_confidence",
    "mechanism_signature",
]

missing_categorical_cols = [col for col in categorical_cols if col not in ctx.columns]
if missing_categorical_cols:
    raise ValueError(
        "Missing required oil categorical column(s):\n"
        + "\n".join(f"- {col}" for col in missing_categorical_cols)
    )

dummy_parts = []
dummy_cols = []

for col in categorical_cols:
    categories = sorted(set(ctx.loc[fit_mask, col].fillna("missing").astype(str)))
    values = ctx[col].fillna("missing").astype(str)

    for category in categories:
        dummy_name = f"{col}_{category}"
        dummy_parts.append(values.eq(category).astype(float).rename(dummy_name))
        dummy_cols.append(dummy_name)

dummies = pd.concat(dummy_parts, axis=1)
ctx = pd.concat([ctx, dummies], axis=1)


# OIL DIFFERENCE:
# Mechanism booleans are oil-specific symbolic predicates from the Qwen/geopolitical extraction.
mechanism_cols = [
    "escalation_active",
    "deescalation_active",
    "supply_risk_active",
    "shipping_or_hormuz_active",
    "sanctions_active",
    "military_action_active",
    "relief_or_demand_weakness_active",
]

missing_mechanism_cols = [col for col in mechanism_cols if col not in ctx.columns]
if missing_mechanism_cols:
    raise ValueError(
        "Missing required oil mechanism column(s):\n"
        + "\n".join(f"- {col}" for col in missing_mechanism_cols)
    )

for col in mechanism_cols:
    ctx[col] = pd.to_numeric(ctx[col], errors="coerce").fillna(0).clip(0, 1)


ctx["llm_confidence_01"] = pd.to_numeric(
    ctx["llm_confidence"],
    errors="coerce",
).fillna(0).clip(0, 1)


# OIL DIFFERENCE:
# Context features are oil-market and conflict-news context, not ticker/SP500 context.
# OIL DIFFERENCE:
# Context features are oil-market and conflict-news context, not MAG7 calendar/ticker context.
context_cols = [
    "prior_up_1h", "prior_down_1h", "prior_abs_1h",
    "prior_up_4h", "prior_down_4h", "prior_abs_4h",
    "prior_up_24h", "prior_down_24h", "prior_abs_24h",
    "quiet_pre_4h", "volatile_pre_4h",
    "quiet_pre_24h", "volatile_pre_24h",
    "event_density_conflict_6h_01",
    "event_density_conflict_24h_01",
    "similar_density_conflict_24h_01",
    "novel_context_24h",
    "hour_sin_01", "hour_cos_01",
]

missing_context_cols = [col for col in context_cols if col not in ctx.columns]
if missing_context_cols:
    raise ValueError(
        "Missing required oil context column(s):\n"
        + "\n".join(f"- {col}" for col in missing_context_cols)
    )
# OIL DIFFERENCE:
# Qwen feature block includes categorical Qwen predicates, mechanism indicators,
# and the model confidence score.
qwen_cols = list(dummies.columns) + mechanism_cols + ["llm_confidence_01"]


# OIL DIFFERENCE:
# Interaction Qwen interactions now combine oil/geopolitical predicates
# with oil pre-event and conflict-news context.
interaction_qwen_cols = [
    c for c in dummy_cols
    if c.startswith("llm_event_direction_")
    or c.startswith("llm_oil_price_direction_")
    or c.startswith("llm_direct_label_")
    or c.startswith("mechanism_signature_")
    or c.startswith("high_confidence_")
] + mechanism_cols

interaction_context_cols = [
    "prior_up_4h",
    "prior_down_4h",
    "quiet_pre_4h",
    "volatile_pre_4h",
    "event_density_conflict_6h_01",
    "event_density_conflict_24h_01",
    "novel_context_24h",
]

interaction_data = {}
for g in interaction_qwen_cols:
    for c in interaction_context_cols:
        name = f"interaction__{g}__x__{c}"
        interaction_data[name] = ctx[g].fillna(0) * ctx[c].fillna(0)

interaction_cols = list(interaction_data.keys())
if interaction_data:
    ctx = pd.concat([ctx, pd.DataFrame(interaction_data, index=ctx.index)], axis=1)


# CHANGED in clean notebook:
# This frame contains oil/Qwen predicates, context features, interactions, and the transfer split only.
# Return columns are kept available, but the final target labels are rebuilt after the return-window audit.
feature_df = ctx.loc[ctx["period"].isin(["training", "validation", "test"])].copy()

for col in qwen_cols + context_cols + interaction_cols:
    feature_df[col] = (
        pd.to_numeric(feature_df[col], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .clip(0, 1)
    )


rule_block_feature_cols = {
    "qwen_semantic": list(qwen_cols),
    "market_context": list(context_cols),
    "qwen_context_interaction": list(interaction_cols),
}

display(feature_df["period"].value_counts().rename("events").to_frame())


# Same experimental shape as MAG7, but using oil-specific feature columns.
variant_specs = {
    "full_ltn_qwen_only": {
        "feature_cols": qwen_cols,
        "rule_blocks": ["qwen_semantic"],
    },
    "full_ltn_context_only": {
        "feature_cols": context_cols,
        "rule_blocks": ["market_context"],
    },
    "full_ltn_qwen_plus_context": {
        "feature_cols": qwen_cols + context_cols,
        "rule_blocks": ["qwen_semantic", "market_context"],
    },
    "full_ltn_qwen_context_interaction": {
        "feature_cols": qwen_cols + context_cols + interaction_cols,
        "rule_blocks": ["qwen_semantic", "market_context", "qwen_context_interaction"],
    },
}

for spec in variant_specs.values():
    spec["feature_cols"] = list(dict.fromkeys(spec["feature_cols"]))
    spec["return_col"] = RETURN_COL

feature_manifest = pd.DataFrame([
    {
        "variant": name,
        "n_features": len(spec["feature_cols"]),
        "rule_blocks": ", ".join(spec["rule_blocks"]),
        "features": ", ".join(spec["feature_cols"]),
    }
    for name, spec in variant_specs.items()
])

if SAVE_OUTPUTS:
    feature_manifest.to_csv(OUTPUT_DIR / "full_ltn_feature_manifest.csv", index=False)

display(feature_manifest)


,events
period,
training,615
validation,410
test,127


,variant,n_features,rule_blocks,features
0,full_ltn_qwen_only,57,qwen_semantic,"llm_event_direction_deescalation, llm_event_di..."
1,full_ltn_context_only,19,market_context,"prior_up_1h, prior_down_1h, prior_abs_1h, prio..."
2,full_ltn_qwen_plus_context,76,"qwen_semantic, market_context","llm_event_direction_deescalation, llm_event_di..."
3,full_ltn_qwen_context_interaction,398,"qwen_semantic, market_context, qwen_context_in...","llm_event_direction_deescalation, llm_event_di..."


## 6. Define Model Variants And LTN Helpers

This section defines the LTN predicates, rule-mining utilities, model-training routine, scoring helper, and summary metrics using the same structure as the MAG7 final notebook.


In [6]:
# Same core model as MAG7:
# This is the full three-label LTN: bearish, bullish, neutral.
# OIL DIFFERENCE:
# The model is trained on oil-specific feature columns and RETURN_COL,
# but the neural/LTN machinery is intentionally kept the same as MAG7.

class OutcomeMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        hidden = max(12, min(80, input_dim * 2))
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.05),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).reshape(-1)


def truth_value(obj):
    return obj.value if hasattr(obj, "value") else obj


def feature_tensor(data, cols):
    x = (
        data[cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .clip(0, 1)
        .to_numpy(dtype=np.float32)
    )
    return torch.tensor(x, dtype=torch.float32, device=device)


def target_tensors(data):
    # Same as MAG7 full-classify:
    # target_label must contain bearish/bullish/neutral labels.
    y = (
        pd.get_dummies(data["target_label"])
        .reindex(columns=LABELS, fill_value=0)
        .to_numpy(dtype=np.float32)
    )

    return {
        "bearish": torch.tensor(y[:, 0].reshape(-1, 1), dtype=torch.float32, device=device),
        "bullish": torch.tensor(y[:, 1].reshape(-1, 1), dtype=torch.float32, device=device),
        "neutral": torch.tensor(y[:, 2].reshape(-1, 1), dtype=torch.float32, device=device),
    }


def label_return_alignment(data, label, return_col):
    # OIL DIFFERENCE:
    # return_col is the selected oil return window rather than MAG7 simple_return.
    # Positive returns are bullish for oil; negative returns are bearish.
    ret = pd.to_numeric(data[return_col], errors="coerce")

    if label == "bullish":
        signed = ret
    elif label == "bearish":
        signed = -ret
    else:
        # For neutral, smaller absolute return is better.
        signed = -ret.abs()

    signed = signed.dropna()

    if len(signed) < 3:
        return np.nan, np.nan

    if label == "neutral":
        p = np.nan
    else:
        p = binomtest(
            int((signed > 0).sum()),
            len(signed),
            0.5,
            alternative="greater",
        ).pvalue

    return float(100 * signed.mean()), float(p) if np.isfinite(p) else np.nan


def empirical_rule_stats(data, condition_cols, label, return_col):
    if len(data) == 0:
        return {
            "n": 0,
            "accuracy_pct": np.nan,
            "aligned_mean_pct": np.nan,
            "aligned_signflip_p": np.nan,
        }

    active = np.ones(len(data), dtype=bool)

    for col in condition_cols:
        active &= (
            pd.to_numeric(data[col], errors="coerce")
            .fillna(0)
            .to_numpy()
            >= ANTECEDENT_THRESHOLD
        )

    subset = data.loc[active].copy()

    if subset.empty:
        return {
            "n": 0,
            "accuracy_pct": np.nan,
            "aligned_mean_pct": np.nan,
            "aligned_signflip_p": np.nan,
        }

    aligned_mean, p = label_return_alignment(subset, label, return_col)

    return {
        "n": int(len(subset)),
        "accuracy_pct": float(100 * subset["target_label"].eq(label).mean()),
        "aligned_mean_pct": aligned_mean,
        "aligned_signflip_p": p,
    }


def infer_rule_block(feature_name):
    # Same block structure as MAG7, but the feature names now refer to oil:
    # qwen_semantic = oil/geopolitical Qwen predicates and mechanism flags
    # market_context = oil pre-event market/conflict-news context
    # qwen_context_interaction = interactions between those two
    if feature_name.startswith("interaction__"):
        return "qwen_context_interaction"

    if feature_name in context_cols:
        return "market_context"

    if feature_name in qwen_cols:
        return "qwen_semantic"

    return "unknown"


def mine_validated_rules(
    data,
    feature_cols,
    return_col,
    rule_blocks=None,
    max_pair_features=36,
    max_rules_per_label=12,
):
    training_df = data.loc[data["period"].eq("training")].copy()
    validation_df = data.loc[data["period"].eq("validation")].copy()

    cols = [c for c in feature_cols if c in data.columns]

    if rule_blocks is not None:
        allowed_blocks = set(rule_blocks)
        cols = [c for c in cols if infer_rule_block(c) in allowed_blocks]

    active_counts = training_df[cols].ge(ANTECEDENT_THRESHOLD).sum().sort_values(ascending=False)
    pair_base = active_counts.head(max_pair_features).index.tolist()

    candidates = [(c,) for c in cols] + list(combinations(pair_base, 2))

    rows = []

    for condition_cols in candidates:
        condition_cols = tuple(condition_cols)

        for label in LABELS:
            disc = empirical_rule_stats(training_df, condition_cols, label, return_col)

            if disc["n"] < MIN_TRAINING_EVENTS or not np.isfinite(disc["accuracy_pct"]):
                continue

            val = empirical_rule_stats(validation_df, condition_cols, label, return_col)

            if val["n"] < MIN_VALIDATION_EVENTS or not np.isfinite(val["accuracy_pct"]):
                continue

            score = (
                disc["accuracy_pct"] / 100
                + val["accuracy_pct"] / 100
                + min(disc["n"] / 40, 1)
                + min(val["n"] / 30, 1)
            )

            if label != "neutral" and np.isfinite(disc["aligned_mean_pct"]):
                score += min(max(disc["aligned_mean_pct"], 0) / 2, 1)

            if label == "neutral" and np.isfinite(disc["aligned_mean_pct"]):
                score += min(max(-disc["aligned_mean_pct"], 0) / 1, 1)

            rows.append({
                "condition": " & ".join(condition_cols),
                "condition_cols_json": json.dumps(condition_cols),
                "rule_label": label,
                "rule_score": score,
                "rule_blocks": ", ".join(sorted({infer_rule_block(c) for c in condition_cols})),
                **{f"training_{k}": v for k, v in disc.items()},
                **{f"validation_{k}": v for k, v in val.items()},
            })

    candidates_df = pd.DataFrame(rows)

    if candidates_df.empty:
        return candidates_df, candidates_df

    selected = candidates_df.loc[
        candidates_df["training_accuracy_pct"].ge(MIN_TRAINING_ACCURACY)
        & candidates_df["validation_accuracy_pct"].ge(MIN_VALIDATION_ACCURACY)
    ].copy()

    selected = selected.sort_values(
        ["rule_label", "validation_accuracy_pct", "validation_n", "rule_score"],
        ascending=[True, False, False, False],
    )

    selected = selected.groupby("rule_label", group_keys=False).head(max_rules_per_label)

    selected = selected.sort_values(
        ["validation_accuracy_pct", "validation_n", "rule_score"],
        ascending=False,
    ).reset_index(drop=True)

    return (
        candidates_df.sort_values("rule_score", ascending=False).reset_index(drop=True),
        selected,
    )


class FeaturePredicate(nn.Module):
    def __init__(self, index):
        super().__init__()
        self.index = index

    def forward(self, z):
        return z[:, self.index].reshape(-1, 1)


def make_feature_predicates(feature_cols):
    preds = {}

    for i, col in enumerate(feature_cols):
        preds[col] = ltn.Predicate(model=FeaturePredicate(i).to(device))

    return preds


def antecedent_from_rule(cols, feat_preds, x, AndOp):
    expr = truth_value(feat_preds[cols[0]](x))

    for col in cols[1:]:
        expr = AndOp(expr, truth_value(feat_preds[col](x)))

    return expr


def train_variant(
    name,
    spec,
    train_df,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=True,
):
    reset_seed(seed)

    feature_cols = spec["feature_cols"]
    return_col = spec["return_col"]

    if use_logic and mine_rules:
        candidates, selected_rules = mine_validated_rules(
            train_df,
            feature_cols,
            return_col,
            rule_blocks=spec.get("rule_blocks"),
            max_pair_features=spec.get("max_pair_features", 36),
            max_rules_per_label=spec.get("max_rules_per_label", 12),
        )

    elif use_logic and not mine_rules:
        candidates = pd.DataFrame()

        if "selected_rules" not in spec:
            raise ValueError(
                f"{name} was called with mine_rules=False, "
                "but spec['selected_rules'] was not provided."
            )

        selected_rules = spec["selected_rules"].copy()

    else:
        candidates = pd.DataFrame()
        selected_rules = pd.DataFrame()

    if SAVE_OUTPUTS and use_logic:
        candidates.to_csv(OUTPUT_DIR / f"{name}_candidate_mined_rules.csv", index=False)
        selected_rules.to_csv(OUTPUT_DIR / f"{name}_selected_validated_rules.csv", index=False)

    X = feature_tensor(train_df, feature_cols)
    x = ltn.Variable(f"x_{name}", X)
    y = target_tensors(train_df)
    feat_preds = make_feature_predicates(feature_cols)

    Bearish = ltn.Predicate(model=OutcomeMLP(X.shape[1]).to(device))
    Bullish = ltn.Predicate(model=OutcomeMLP(X.shape[1]).to(device))
    Neutral = ltn.Predicate(model=OutcomeMLP(X.shape[1]).to(device))

    AndOp = ltn.fuzzy_ops.AndProd()
    ImpliesOp = ltn.fuzzy_ops.ImpliesReichenbach()
    ForallAgg = ltn.fuzzy_ops.AggregPMeanError(p=2)
    SatAgg = ltn.fuzzy_ops.SatAgg()

    label_predicates = {
        "bearish": Bearish,
        "bullish": Bullish,
        "neutral": Neutral,
    }

    rule_specs = []

    for _, rule in selected_rules.iterrows():
        cols = [c for c in json.loads(rule["condition_cols_json"]) if c in feat_preds]

        if cols:
            rule_specs.append((rule["condition"], cols, rule["rule_label"]))

    params = (
        list(Bearish.model.parameters())
        + list(Bullish.model.parameters())
        + list(Neutral.model.parameters())
    )

    opt = torch.optim.Adam(params, lr=LR)

    history = []

    for epoch in range(EPOCHS):
        opt.zero_grad()

        pred_bear = truth_value(Bearish(x)).reshape(-1, 1)
        pred_bull = truth_value(Bullish(x)).reshape(-1, 1)
        pred_neut = truth_value(Neutral(x)).reshape(-1, 1)

        supervised = (
            F.mse_loss(pred_bear, y["bearish"])
            + F.mse_loss(pred_bull, y["bullish"])
            + F.mse_loss(pred_neut, y["neutral"])
        )

        exclusivity = torch.mean(
            pred_bear * pred_bull
            + pred_bear * pred_neut
            + pred_bull * pred_neut
        )

        formulas = []

        if use_logic:
            for rule_name, cols, label in rule_specs:
                formulas.append((
                    rule_name,
                    ForallAgg(
                        ImpliesOp(
                            antecedent_from_rule(cols, feat_preds, x, AndOp),
                            truth_value(label_predicates[label](x)),
                        )
                    ),
                ))

        sat = (
            SatAgg(*[formula for _, formula in formulas])
            if formulas
            else torch.tensor(1.0, device=device)
        )

        logic_penalty = (
            logic_weight * (1.0 - truth_value(sat))
            if use_logic
            else torch.tensor(0.0, device=device)
        )

        loss = supervised + 0.10 * exclusivity + logic_penalty
        loss.backward()
        opt.step()

        if epoch % 50 == 0 or epoch == EPOCHS - 1:
            row = {
                "variant": name,
                "epoch": epoch,
                "seed": seed,
                "use_logic": use_logic,
                "mine_rules": mine_rules,
                "logic_weight": logic_weight,
                "loss": float(loss.detach().cpu()),
                "supervised_mse": float(supervised.detach().cpu()),
                "exclusivity": float(exclusivity.detach().cpu()),
                "sat": float(truth_value(sat).detach().cpu()) if hasattr(truth_value(sat), "detach") else float(sat),
                "n_selected_rules": len(rule_specs),
            }

            for rule_name, formula in formulas[:20]:
                row[f"sat_{rule_name[:80]}"] = float(truth_value(formula).detach().cpu())

            history.append(row)

    return {
        "name": name,
        "seed": seed,
        "feature_cols": feature_cols,
        "candidate_rules": candidates,
        "selected_rules": selected_rules,
        "use_logic": use_logic,
        "mine_rules": mine_rules,
        "logic_weight": logic_weight,
        "Bearish": Bearish,
        "Bullish": Bullish,
        "Neutral": Neutral,
        "history": pd.DataFrame(history),
    }

In [7]:
# Same scoring/evaluation structure as MAG7.
# OIL DIFFERENCE:
# aligned_stats and summary_row use the selected oil RETURN_COL,
# instead of MAG7 simple_return.

def score_variant(bundle, df):
    out = df.copy()
    X = feature_tensor(out, bundle["feature_cols"])

    with torch.no_grad():
        eval_x = ltn.Variable(f"eval_{bundle['name']}", X)

        out["score_bearish"] = (
            truth_value(bundle["Bearish"](eval_x))
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )
        out["score_bullish"] = (
            truth_value(bundle["Bullish"](eval_x))
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )
        out["score_neutral"] = (
            truth_value(bundle["Neutral"](eval_x))
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )

    scores = out[["score_bearish", "score_bullish", "score_neutral"]].to_numpy()

    out["prediction"] = np.array(LABELS)[scores.argmax(axis=1)]
    out["confidence_margin"] = (
        np.sort(scores, axis=1)[:, -1]
        - np.sort(scores, axis=1)[:, -2]
    )

    return out


def aligned_stats(df):
    # OIL DIFFERENCE:
    # MAG7 used simple_return. Oil uses the selected RETURN_COL.
    ret = pd.to_numeric(df[RETURN_COL], errors="coerce")
    pred = df["prediction"]

    signed = np.select(
        [pred.eq("bullish"), pred.eq("bearish")],
        [ret, -ret],
        default=np.nan,
    )

    signed = pd.Series(signed, index=df.index).dropna()

    if len(signed) < 3:
        return np.nan, np.nan

    p = binomtest(
        int((signed > 0).sum()),
        len(signed),
        0.5,
        alternative="greater",
    ).pvalue

    return float(signed.mean() * 100), float(p)


def summary_row(variant, period, df, bundle=None):
    aligned_mean, aligned_p = aligned_stats(df)

    row = {
        "variant": variant,
        "period": period,
        "n": int(len(df)),
        "accuracy_pct": float(100 * df["prediction"].eq(df["target_label"]).mean()),

        # OIL DIFFERENCE:
        # Keep generic naming because oil uses RETURN_COL, not MAG7 simple_return.
        "mean_return_pct": float(100 * pd.to_numeric(df[RETURN_COL], errors="coerce").mean()),

        "aligned_mean_pct": aligned_mean,
        "aligned_signflip_p": aligned_p,
        "bullish_pred_rate_pct": float(100 * df["prediction"].eq("bullish").mean()),
        "bearish_pred_rate_pct": float(100 * df["prediction"].eq("bearish").mean()),
        "neutral_pred_rate_pct": float(100 * df["prediction"].eq("neutral").mean()),
        "mean_confidence_margin": float(df["confidence_margin"].mean()),
    }

    if bundle is not None:
        row["use_logic"] = bundle["use_logic"]
        row["logic_weight"] = bundle["logic_weight"]
        row["n_selected_rules"] = len(bundle["selected_rules"])

    return row


## 7. Validation-Only Return-Window Audit

This audit compares candidate oil return windows using training and validation data only. The held-out Iran test period is excluded from this step.


In [8]:
# CHANGED in clean notebook:
# The return-window audit now runs before the final RETURN_COL model_df is constructed.
# It uses the horizon-independent feature_df and rebuilds target labels for each candidate return column.
# Training + validation only are used here; the held-out Iran test period is not used for return-window choice.

RETURN_WINDOW_AUDIT_VARIANT = "full_ltn_qwen_plus_context"

available_return_candidates = [
    (window, col)
    for window, col in RETURN_WINDOW_CANDIDATES
    if col in feature_df.columns
]

if not available_return_candidates:
    raise ValueError(
        "None of the requested oil return-window columns exist in feature_df. "
        "Check the available ret_* columns in the aligned oil event-return parquet."
    )

print("Available oil return windows:", available_return_candidates)


def build_model_df_for_return_col(return_col: str) -> pd.DataFrame:
    out = feature_df.copy()

    missing_returns = int(out[return_col].isna().sum())
    if missing_returns:
        print(f"{return_col}: dropping {missing_returns} rows with missing returns")
        out = out.loc[out[return_col].notna()].copy()

    out["target_label"] = out[return_col].map(
        lambda x: realised_label_from_return(float(x), RETURN_THRESHOLD)
    )

    return out


return_window_ltn_rows = []


def summary_row_for_return_col(variant, period, df, return_col, bundle=None):
    ret = pd.to_numeric(df[return_col], errors="coerce")
    pred = df["prediction"]

    signed = np.select(
        [pred.eq("bullish"), pred.eq("bearish")],
        [ret, -ret],
        default=np.nan,
    )
    signed = pd.Series(signed, index=df.index).dropna()

    if len(signed) >= 3:
        aligned_mean = float(signed.mean() * 100)
        aligned_p = float(
            binomtest(
                int((signed > 0).sum()),
                len(signed),
                0.5,
                alternative="greater",
            ).pvalue
        )
    else:
        aligned_mean = np.nan
        aligned_p = np.nan

    row = {
        "variant": variant,
        "period": period,
        "n": int(len(df)),
        "accuracy_pct": float(100 * df["prediction"].eq(df["target_label"]).mean()),
        "mean_return_pct": float(100 * ret.mean()),
        "aligned_mean_pct": aligned_mean,
        "aligned_signflip_p": aligned_p,
        "bullish_pred_rate_pct": float(100 * df["prediction"].eq("bullish").mean()),
        "bearish_pred_rate_pct": float(100 * df["prediction"].eq("bearish").mean()),
        "neutral_pred_rate_pct": float(100 * df["prediction"].eq("neutral").mean()),
        "mean_confidence_margin": float(df["confidence_margin"].mean()),
    }

    if bundle is not None:
        row["use_logic"] = bundle["use_logic"]
        row["logic_weight"] = bundle["logic_weight"]
        row["n_selected_rules"] = len(bundle["selected_rules"])

    return row

for return_window, return_col in available_return_candidates:
    print(f"Oil LTN return-window audit: {return_window} using {return_col}")

    model_df_h = build_model_df_for_return_col(return_col)

    train_h = model_df_h.loc[model_df_h["period"].eq("training")].copy()
    validation_h = model_df_h.loc[model_df_h["period"].eq("validation")].copy()
    train_val_h = model_df_h.loc[
        model_df_h["period"].isin(["training", "validation"])
    ].copy()

    spec_base_h = variant_specs[RETURN_WINDOW_AUDIT_VARIANT].copy()
    spec_base_h["return_col"] = return_col

    candidate_rules_h, selected_rules_h = mine_validated_rules(
        train_val_h,
        spec_base_h["feature_cols"],
        spec_base_h["return_col"],
        rule_blocks=spec_base_h.get("rule_blocks"),
        max_pair_features=spec_base_h.get("max_pair_features", 36),
        max_rules_per_label=spec_base_h.get("max_rules_per_label", 12),
    )

    spec_h = spec_base_h.copy()
    spec_h["selected_rules"] = selected_rules_h.copy()

    bundle_h = train_variant(
        f"{RETURN_WINDOW_AUDIT_VARIANT}_{return_window}",
        spec_h,
        train_h,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=SEED,
        mine_rules=False,
    )

    scored_validation_h = score_variant(bundle_h, validation_h)

    row_h = summary_row_for_return_col(
        f"LTN return window {return_window}",
        "validation",
        scored_validation_h,
        return_col,
        bundle=bundle_h,
    )

    majority_baseline_h = validation_h["target_label"].value_counts(normalize=True).max() * 100

    row_h["return_window"] = return_window
    row_h["return_col"] = return_col
    row_h["training_n"] = len(train_h)
    row_h["validation_n"] = len(validation_h)
    row_h["validation_majority_baseline_pct"] = float(majority_baseline_h)
    row_h["validation_bullish_n"] = int(validation_h["target_label"].eq("bullish").sum())
    row_h["validation_bearish_n"] = int(validation_h["target_label"].eq("bearish").sum())
    row_h["validation_neutral_n"] = int(validation_h["target_label"].eq("neutral").sum())
    row_h["n_candidate_rules"] = len(candidate_rules_h)
    row_h["n_selected_rules"] = len(selected_rules_h)

    return_window_ltn_rows.append(row_h)


return_window_ltn_audit = pd.DataFrame(return_window_ltn_rows)

display(
    return_window_ltn_audit.sort_values(
        ["accuracy_pct", "aligned_mean_pct", "n_selected_rules"],
        ascending=False,
    )
)


Available oil return windows: [('1h', 'ret_1h'), ('4h', 'ret_4h'), ('24h', 'ret_24h')]
Oil LTN return-window audit: 1h using ret_1h
ret_1h: dropping 1 rows with missing returns
Oil LTN return-window audit: 4h using ret_4h
ret_4h: dropping 2 rows with missing returns
Oil LTN return-window audit: 24h using ret_24h


,variant,period,n,accuracy_pct,mean_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules,return_window,return_col,training_n,validation_n,validation_majority_baseline_pct,validation_bullish_n,validation_bearish_n,validation_neutral_n,n_candidate_rules
2,LTN return window 24h,validation,410,49.268293,-0.339819,-0.127978,0.635195,43.902439,56.097561,0.0,0.677042,True,0.3,23,24h,ret_24h,615,410,53.658537,189,220,1,1764
0,LTN return window 1h,validation,409,46.699267,-0.019681,-0.038171,0.916938,49.877751,50.122249,0.0,0.770034,True,0.3,19,1h,ret_1h,615,409,51.589242,197,211,1,1761
1,LTN return window 4h,validation,408,46.323529,-0.066534,-0.157839,0.937627,54.901961,45.098039,0.0,0.718054,True,0.3,14,4h,ret_4h,615,408,51.225490,199,209,0,1764


### 8. Select Primary Return Window And Build Final Model Frame

After the return-window audit, the retained `RETURN_COL` is attached to the feature frame to create the final `model_df`. This is the frame used for frozen-rule refitting and held-out Iran testing.


In [9]:
# Based on validation accuracy, retain the selected oil return window.
RETURN_COL = "ret_24h"


In [10]:
# CHANGED in clean notebook:
# Construct the final modelling frame only after the validation-only return-window audit.
# RETURN_COL remains the retained primary return window for the final held-out Iran evaluation.

model_df = build_model_df_for_return_col(RETURN_COL)

for spec in variant_specs.values():
    spec["return_col"] = RETURN_COL

train = model_df.loc[model_df["period"].eq("training")].copy()
validation = model_df.loc[model_df["period"].eq("validation")].copy()
test = model_df.loc[model_df["period"].eq("test")].copy()

display(model_df.groupby("period")["target_label"].value_counts().unstack(fill_value=0))


target_label,bearish,bullish,neutral
period,,,
test,89,38,0
training,289,326,0
validation,220,189,1


## 9. Mine/Select Rules, Freeze Them, Then Refit

Candidate rules are generated from the training period and filtered using validation-period evidence. The selected rules are frozen, then each LTN feature variant is refit on training plus validation and evaluated once on the held-out Iran test set.

### LTN variants experiment 1

This experiment compares LTN feature variants after the primary oil return window has been selected.


In [11]:
train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()

frozen_variant_specs = {}

for name, spec in variant_specs.items():
    print("selecting frozen rules", name)

    candidate_rules, selected_rules = mine_validated_rules(
        train_val,
        spec["feature_cols"],
        spec["return_col"],
        rule_blocks=spec.get("rule_blocks"),
        max_pair_features=spec.get("max_pair_features", 36),
        max_rules_per_label=spec.get("max_rules_per_label", 12),
    )

    frozen_spec = spec.copy()
    frozen_spec["selected_rules"] = selected_rules.copy()
    frozen_variant_specs[name] = frozen_spec


trained_final = {}

for name, spec in frozen_variant_specs.items():
    print("final frozen-rule training", name, "rows", len(train_val))

    trained_final[name] = train_variant(
        name,
        spec,
        train_val,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=SEED,
        mine_rules=False,
    )


final_rows = []
final_frames = []

for variant, bundle in trained_final.items():
    scored = score_variant(bundle, test)

    final_rows.append(
        summary_row(
            variant,
            "iran_test_frozen_rules_refit_training_validation",
            scored,
            bundle=bundle,
        )
    )

    final_frames.append(
        scored.assign(
            variant=variant,
            period="iran_test_frozen_rules_refit_training_validation",
        )
    )


final_summary = pd.DataFrame(final_rows)

if SAVE_OUTPUTS:
    final_summary.to_csv(
        OUTPUT_DIR / "full_ltn_final_frozen_rule_iran_test_summary.csv",
        index=False,
    )

    pd.concat(final_frames, ignore_index=True).to_csv(
        OUTPUT_DIR / "full_ltn_final_frozen_rule_iran_test_predictions.csv",
        index=False,
    )


display(
    final_summary.sort_values(
        ["accuracy_pct", "aligned_mean_pct"],
        ascending=False,
    )
)


selecting frozen rules full_ltn_qwen_only
selecting frozen rules full_ltn_context_only
selecting frozen rules full_ltn_qwen_plus_context
selecting frozen rules full_ltn_qwen_context_interaction
final frozen-rule training full_ltn_qwen_only rows 1025
final frozen-rule training full_ltn_context_only rows 1025
final frozen-rule training full_ltn_qwen_plus_context rows 1025
final frozen-rule training full_ltn_qwen_context_interaction rows 1025


,variant,period,n,accuracy_pct,mean_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
2,full_ltn_qwen_plus_context,iran_test_frozen_rules_refit_training_validation,127,59.842520,-1.491011,0.598495,0.016395,27.559055,72.440945,0.0,0.698626,True,0.3,23
1,full_ltn_context_only,iran_test_frozen_rules_refit_training_validation,127,55.905512,-1.491011,0.572111,0.106983,51.968504,48.031496,0.0,0.528875,True,0.3,12
0,full_ltn_qwen_only,iran_test_frozen_rules_refit_training_validation,127,55.905512,-1.491011,0.169770,0.106983,37.795276,62.204724,0.0,0.623222,True,0.3,24
3,full_ltn_qwen_context_interaction,iran_test_frozen_rules_refit_training_validation,127,49.606299,-1.491011,0.031974,0.570386,39.370079,60.629921,0.0,0.725041,True,0.3,14


## 10. LTN vs no LTN ablation test 1

This section compares the LTN Qwen-plus-context model against a neural classifier using the same feature set with logic disabled, using the standard oil transfer test frame.


In [12]:
# LTN vs no-logic ablation.
#
# Same purpose as MAG7:
# Compare the full LTN rule-constrained model against the same neural scorer
# with Qwen + context features but with the logic/rule component disabled.
#
# OIL DIFFERENCE:
# train_val = Russia/Ukraine training + validation.
# test = Iran held-out transfer set.

train_val = model_df.loc[
    model_df["period"].isin(["training", "validation"])
].copy()

test = model_df.loc[
    model_df["period"].eq("test")
].copy()

ablation_base_spec = variant_specs["full_ltn_qwen_plus_context"]


candidate_rules, selected_rules = mine_validated_rules(
    train_val,
    ablation_base_spec["feature_cols"],
    ablation_base_spec["return_col"],
    rule_blocks=ablation_base_spec.get("rule_blocks"),
    max_pair_features=ablation_base_spec.get("max_pair_features", 36),
    max_rules_per_label=ablation_base_spec.get("max_rules_per_label", 12),
)

ablation_ltn_spec = ablation_base_spec.copy()
ablation_ltn_spec["selected_rules"] = selected_rules.copy()

# Rule-support diagnostic omitted to match the MAG7 final notebook structure.



ltn_bundle = train_variant(
    "oil_ltn_qwen_plus_context_ablation",
    ablation_ltn_spec,
    train_val,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=False,
)

ltn_scored = score_variant(ltn_bundle, test)


no_logic_bundle = train_variant(
    "oil_no_logic_qwen_plus_context_ablation",
    ablation_base_spec,
    train_val,
    use_logic=False,
    logic_weight=0.0,
    seed=SEED,
    mine_rules=False,
)

no_logic_scored = score_variant(no_logic_bundle, test)


ltn_ablation_summary = pd.DataFrame([
    summary_row(
        "LTN, Qwen + context",
        "iran_heldout_test",
        ltn_scored,
        bundle=ltn_bundle,
    ),
    summary_row(
        "Neural classifier, Qwen + context, no logic",
        "iran_heldout_test",
        no_logic_scored,
        bundle=no_logic_bundle,
    ),
])

display(ltn_ablation_summary)


if SAVE_OUTPUTS:
    ltn_bundle["history"].to_csv(
        OUTPUT_DIR / "oil_ltn_qwen_plus_context_ablation_training_history.csv",
        index=False,
    )

    no_logic_bundle["history"].to_csv(
        OUTPUT_DIR / "oil_no_logic_qwen_plus_context_ablation_training_history.csv",
        index=False,
    )

    ltn_scored.to_csv(
        OUTPUT_DIR / "oil_ltn_qwen_plus_context_ablation_iran_test_predictions.csv",
        index=False,
    )

    no_logic_scored.to_csv(
        OUTPUT_DIR / "oil_no_logic_qwen_plus_context_ablation_iran_test_predictions.csv",
        index=False,
    )

    ltn_ablation_summary.to_csv(
        OUTPUT_DIR / "oil_ltn_ablation_qwen_plus_context_iran_test_summary.csv",
        index=False,
    )

,variant,period,n,accuracy_pct,mean_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
0,"LTN, Qwen + context",iran_heldout_test,127,55.905512,-1.491011,0.297311,0.106983,28.346457,71.653543,0.0,0.690214,True,0.3,23
1,"Neural classifier, Qwen + context, no logic",iran_heldout_test,127,57.480315,-1.491011,0.551874,0.054925,37.795276,62.204724,0.0,0.641345,False,0.0,0


## 11. LTN vs no LTN ablation test 2

Same ablation experiment as test one, but over repeated seeds using the standard oil transfer test frame.


In [13]:
ROBUST_SEEDS = [1, 2, 3, 4, 5, 7, 11, 13, 17, 19]

In [14]:
# Main repeated-seed ablation for the oil case study.
#
# Oil difference from MAG7:
# - This keeps the fixed transfer setup:
#     Russia/Ukraine training + validation -> Iran held-out test
# - It does not create a longer calendar test window.
# - The purpose is to check whether the LTN vs no-logic result is stable
#   across random neural initialisations.

seed_rows = []
seed_rule_rows = []

train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()
test = model_df.loc[model_df["period"].eq("test")].copy()

base_spec = variant_specs["full_ltn_qwen_plus_context"]

for seed in ROBUST_SEEDS:
    print(f"Main repeated-seed ablation: seed {seed}")

    # Rules are mined using training + validation only.
    # Oil difference from MAG7:
    # - Iran test rows are not used for rule selection.
    # - This preserves the conflict-transfer evaluation.
    candidate_rules, selected_rules = mine_validated_rules(
        train_val,
        base_spec["feature_cols"],
        base_spec["return_col"],
        rule_blocks=base_spec.get("rule_blocks"),
        max_pair_features=base_spec.get("max_pair_features", 36),
        max_rules_per_label=base_spec.get("max_rules_per_label", 12),
    )

    ltn_spec = base_spec.copy()
    ltn_spec["selected_rules"] = selected_rules.copy()

    seed_rule_rows.append({
        "seed": seed,
        "model": "LTN, Qwen + context",
        "rule_blocks": ", ".join(base_spec.get("rule_blocks", [])),
        "n_candidate_rules": len(candidate_rules),
        "n_selected_rules": len(selected_rules),
        "validation_n_min": selected_rules["validation_n"].min() if len(selected_rules) else np.nan,
        "validation_n_median": selected_rules["validation_n"].median() if len(selected_rules) else np.nan,
        "validation_n_max": selected_rules["validation_n"].max() if len(selected_rules) else np.nan,
        "validation_accuracy_mean": selected_rules["validation_accuracy_pct"].mean() if len(selected_rules) else np.nan,
    })

    # LTN branch uses frozen selected rules, so mine_rules=False.
    ltn_bundle = train_variant(
        f"oil_ltn_qwen_plus_context_seed_{seed}",
        ltn_spec,
        train_val,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=seed,
        mine_rules=False,
    )

    ltn_scored = score_variant(ltn_bundle, test)

    seed_rows.append({
        "seed": seed,
        "model": "LTN, Qwen + context",
        **summary_row(
            "LTN, Qwen + context",
            "iran_heldout_test",
            ltn_scored,
            bundle=ltn_bundle,
        ),
    })

    # No-logic branch keeps the same neural inputs but disables symbolic formulas.
    no_logic_bundle = train_variant(
        f"oil_no_logic_qwen_plus_context_seed_{seed}",
        base_spec,
        train_val,
        use_logic=False,
        logic_weight=0.0,
        seed=seed,
        mine_rules=False,
    )

    no_logic_scored = score_variant(no_logic_bundle, test)

    seed_rows.append({
        "seed": seed,
        "model": "Neural classifier, Qwen + context, no logic",
        **summary_row(
            "Neural classifier, Qwen + context, no logic",
            "iran_heldout_test",
            no_logic_scored,
            bundle=no_logic_bundle,
        ),
    })


seed_results = pd.DataFrame(seed_rows)
seed_rule_summary = pd.DataFrame(seed_rule_rows)

seed_summary = (
    seed_results
    .groupby("model")
    .agg(
        runs=("seed", "nunique"),
        accuracy_mean=("accuracy_pct", "mean"),
        accuracy_std=("accuracy_pct", "std"),
        aligned_mean_return_mean=("aligned_mean_pct", "mean"),
        aligned_mean_return_std=("aligned_mean_pct", "std"),
        confidence_margin_mean=("mean_confidence_margin", "mean"),
        confidence_margin_std=("mean_confidence_margin", "std"),
        selected_rules_mean=("n_selected_rules", "mean"),
        selected_rules_min=("n_selected_rules", "min"),
        selected_rules_max=("n_selected_rules", "max"),
    )
    .reset_index()
)

display(seed_results)
display(seed_summary)


if SAVE_OUTPUTS:
    seed_results.to_csv(
        OUTPUT_DIR / "oil_repeated_seed_ablation_iran_test_results.csv",
        index=False,
    )

    seed_summary.to_csv(
        OUTPUT_DIR / "oil_repeated_seed_ablation_iran_test_summary.csv",
        index=False,
    )

Main repeated-seed ablation: seed 1
Main repeated-seed ablation: seed 2
Main repeated-seed ablation: seed 3
Main repeated-seed ablation: seed 4
Main repeated-seed ablation: seed 5
Main repeated-seed ablation: seed 7
Main repeated-seed ablation: seed 11
Main repeated-seed ablation: seed 13
Main repeated-seed ablation: seed 17
Main repeated-seed ablation: seed 19


,seed,model,variant,period,n,accuracy_pct,mean_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,neutral_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
0,1,"LTN, Qwen + context","LTN, Qwen + context",iran_heldout_test,127,52.755906,-1.491011,0.147487,0.297308,37.795276,62.204724,0.0,0.707974,True,0.3,23
1,1,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",iran_heldout_test,127,54.330709,-1.491011,0.230059,0.187479,40.944882,59.055118,0.0,0.704413,False,0.0,0
2,2,"LTN, Qwen + context","LTN, Qwen + context",iran_heldout_test,127,65.354331,-1.491011,0.779222,0.000342,33.070866,66.929134,0.0,0.744373,True,0.3,23
3,2,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",iran_heldout_test,127,55.905512,-1.491011,0.242755,0.106983,37.795276,62.204724,0.0,0.723084,False,0.0,0
4,3,"LTN, Qwen + context","LTN, Qwen + context",iran_heldout_test,127,51.968504,-1.491011,0.335025,0.361394,35.433071,64.566929,0.0,0.687935,True,0.3,23
5,3,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",iran_heldout_test,127,58.267717,-1.491011,0.484250,0.037765,38.582677,61.417323,0.0,0.673082,False,0.0,0
6,4,"LTN, Qwen + context","LTN, Qwen + context",iran_heldout_test,127,53.543307,-1.491011,0.207860,0.238961,35.433071,64.566929,0.0,0.713340,True,0.3,23
7,4,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",iran_heldout_test,127,50.393701,-1.491011,-0.124565,0.500000,43.307087,56.692913,0.0,0.732898,False,0.0,0
8,5,"LTN, Qwen + context","LTN, Qwen + context",iran_heldout_test,127,56.692913,-1.491011,0.550330,0.077700,33.858268,66.141732,0.0,0.719206,True,0.3,23
9,5,"Neural classifier, Qwen + context, no logic","Neural classifier, Qwen + context, no logic",iran_heldout_test,127,53.543307,-1.491011,0.148088,0.238961,40.157480,59.842520,0.0,0.737745,False,0.0,0


,model,runs,accuracy_mean,accuracy_std,aligned_mean_return_mean,aligned_mean_return_std,confidence_margin_mean,confidence_margin_std,selected_rules_mean,selected_rules_min,selected_rules_max
0,"LTN, Qwen + context",10,55.590551,4.668671,0.389310,0.253337,0.708665,0.026728,23.0,23,23
1,"Neural classifier, Qwen + context, no logic",10,53.700787,3.210265,0.265151,0.216947,0.708921,0.030726,0.0,0,0
